TEXT PREPROCESSING

incremental text preprocessing in three steps: 

1.
2.
3.

In [2]:
#setup

from pathlib import Path
import string
import pandas as pd

from nltk.tokenize import wordpunct_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

pd.set_option("display.max_colwidth", 120)


DATA_PATH = Path("../data/Sentences_50Agree.txt")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


rows = []

with open(DATA_PATH, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if line:
            text, label = line.rsplit("@", 1)
            rows.append({
                "text": text.strip(),
                "label": label.strip()
            })

df = pd.DataFrame(rows)

print("Dataset loaded successfully.")
print("Number of examples:", len(df))
print("Labels:", sorted(df["label"].unique()))

df.head()

Dataset loaded successfully.
Number of examples: 4846
Labels: ['negative', 'neutral', 'positive']


,text,label
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",neutral
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...",neutral
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,negative
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,positive
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...",positive


In [3]:
stop_words = set(ENGLISH_STOP_WORDS)

words_to_keep = {
    "no", "not", "nor",
    "up", "down",
}

stop_words = stop_words - words_to_keep

print("Number of stop words:", len(stop_words))

Number of stop words: 313


In [15]:
#1) lowercase and remove extra whitespace

def preprocess_minimal(text):
    text = text.lower()
    text = " ".join(text.split())
    return text

df["text_minimal"] = df["text"].apply(preprocess_minimal)

df[["text", "text_minimal"]].head()

,text,text_minimal
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...","according to gran , the company has no plans to move all production to russia , although that is where the company i..."
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...","technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work..."
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,the international electronic industry company elcoteq has laid off tens of employees from its tallinn facility ; con...
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,with the new production plant the company would increase its capacity to meet the expected increase in demand and wo...
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...","according to the company 's updated strategy for the years 2009-2012 , basware targets a long-term net sales growth ..."


In [16]:
#2) remove punctuation

def preprocess_no_punctuation(text):
    tokens = wordpunct_tokenize(text.lower())

    cleaned_tokens = [
        token for token in tokens
        if token not in string.punctuation
    ]

    return " ".join(cleaned_tokens)

df["text_no_punctuation"] = df["text"].apply(preprocess_no_punctuation)

df[["text", "text_no_punctuation"]].head()

,text,text_no_punctuation
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",according to gran the company has no plans to move all production to russia although that is where the company is gr...
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...",technopolis plans to develop in stages an area of no less than 100 000 square meters in order to host companies work...
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,the international electronic industry company elcoteq has laid off tens of employees from its tallinn facility contr...
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,with the new production plant the company would increase its capacity to meet the expected increase in demand and wo...
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...",according to the company s updated strategy for the years 2009 2012 basware targets a long term net sales growth in ...


In [17]:
#3) remove stop words

def preprocess_stopwords(text):
    tokens = wordpunct_tokenize(text.lower())

    cleaned_tokens = [
        token for token in tokens
        if token not in string.punctuation
        and token not in stop_words
    ]

    return " ".join(cleaned_tokens)

df["text_stopwords"] = df["text"].apply(preprocess_stopwords)

df[["text", "text_stopwords"]].head()

,text,text_stopwords
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",according gran company no plans production russia company growing
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...",technopolis plans develop stages area no 100 000 square meters order host companies working computer technologies te...
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,international electronic industry company elcoteq laid tens employees tallinn facility contrary earlier layoffs comp...
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,new production plant company increase capacity meet expected increase demand improve use raw materials increase prod...
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...",according company s updated strategy years 2009 2012 basware targets long term net sales growth range 20 40 operatin...


In [18]:
#4) stemming

stemmer = PorterStemmer()

def preprocess_stemming(text):
    tokens = wordpunct_tokenize(text.lower())

    cleaned_tokens = [
        stemmer.stem(token)
        for token in tokens
        if token not in string.punctuation
        and token not in stop_words
    ]

    return " ".join(cleaned_tokens)

df["text_stemming"] = df["text"].apply(preprocess_stemming)

df[["text", "text_stemming"]].head()

,text,text_stemming
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",accord gran compani no plan product russia compani grow
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...",technopoli plan develop stage area no 100 000 squar meter order host compani work comput technolog telecommun statem...
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,intern electron industri compani elcoteq laid ten employe tallinn facil contrari earlier layoff compani contract ran...
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,new product plant compani increas capac meet expect increas demand improv use raw materi increas product profit
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...",accord compani s updat strategi year 2009 2012 baswar target long term net sale growth rang 20 40 oper profit margin...


In [19]:
#examples

comparison_examples = df[
    ["text", "text_minimal", "text_no_punctuation", "text_stopwords", "text_stemming", "label"]
].sample(5, random_state=42)

comparison_examples

,text,text_minimal,text_no_punctuation,text_stopwords,text_stemming,label
3207,"The company was supposed to deliver machinery to a veneer mill in the Tomsk region , in Russia .","the company was supposed to deliver machinery to a veneer mill in the tomsk region , in russia .",the company was supposed to deliver machinery to a veneer mill in the tomsk region in russia,company supposed deliver machinery veneer tomsk region russia,compani suppos deliv machineri veneer tomsk region russia,neutral
1684,UNC Charlotte would also deploy SSH Tectia Connector to enable secure application connectivity .,unc charlotte would also deploy ssh tectia connector to enable secure application connectivity .,unc charlotte would also deploy ssh tectia connector to enable secure application connectivity,unc charlotte deploy ssh tectia connector enable secure application connectivity,unc charlott deploy ssh tectia connector enabl secur applic connect,neutral
1044,"In 2009 , Lee & Man had a combined annual production capacity of close to 4.5 million tonnes of paper and 300,000 to...","in 2009 , lee & man had a combined annual production capacity of close to 4.5 million tonnes of paper and 300,000 to...",in 2009 lee man had a combined annual production capacity of close to 4 5 million tonnes of paper and 300 000 tonnes...,2009 lee man combined annual production capacity close 4 5 million tonnes paper 300 000 tonnes pulp,2009 lee man combin annual product capac close 4 5 million tonn paper 300 000 tonn pulp,neutral
4145,"`` That 's a very high figure on the European scale , '' Noop said , recalling however that this also includes beer ...","`` that 's a very high figure on the european scale , '' noop said , recalling however that this also includes beer ...",`` that s a very high figure on the european scale '' noop said recalling however that this also includes beer bough...,`` s high figure european scale '' noop said recalling includes beer bought finnish tourists,`` s high figur european scale '' noop said recal includ beer bought finnish tourist,neutral
1538,"In Finland , the corresponding service is Alma Media 's Etuovi.com , Finland 's most popular and best known nationwi...","in finland , the corresponding service is alma media 's etuovi.com , finland 's most popular and best known nationwi...",in finland the corresponding service is alma media s etuovi com finland s most popular and best known nationwide onl...,finland corresponding service alma media s etuovi com finland s popular best known nationwide online service home pr...,finland correspond servic alma media s etuovi com finland s popular best known nationwid onlin servic home properti ...,neutral


In [20]:
#comparison

preprocessing_columns = [
    "text_minimal",
    "text_no_punctuation",
    "text_stopwords",
    "text_stemming"
]

summary_rows = []

for column in preprocessing_columns:
    tokenized_texts = df[column].apply(wordpunct_tokenize)

    all_tokens = [
        token
        for tokens in tokenized_texts
        for token in tokens
    ]

    vocabulary = set(all_tokens)

    summary_rows.append({
        "preprocessing": column,
        "avg_tokens_per_headline": round(tokenized_texts.apply(len).mean(), 2),
        "vocabulary_size": len(vocabulary),
        "total_tokens": len(all_tokens)
    })

summary = pd.DataFrame(summary_rows)

summary

,preprocessing,avg_tokens_per_headline,vocabulary_size,total_tokens
0,text_minimal,25.10,10151,121652
1,text_no_punctuation,21.42,10134,103825
2,text_stopwords,13.88,9887,67245
3,text_stemming,13.88,7669,67245


In [21]:
selected_preprocessing = {
    "minimal": "text_minimal",
    "no_punctuation": "text_no_punctuation",
    "stopwords": "text_stopwords",
    "stemming": "text_stemming"
}

selected_preprocessing

{'minimal': 'text_minimal',
 'no_punctuation': 'text_no_punctuation',
 'stopwords': 'text_stopwords',
 'stemming': 'text_stemming'}

In [22]:
X_minimal = df["text_minimal"]
X_no_punctuation = df["text_no_punctuation"]
X_stopwords = df["text_stopwords"]
X_stemming = df["text_stemming"]
y = df["label"]

In [23]:
task2_output_path = OUTPUT_DIR / "preprocessed_texts.csv"

df[
    ["text", "label", "text_minimal", "text_no_punctuation", "text_stopwords", "text_stemming"]
].to_csv(task2_output_path, index=False)

print("Saved preprocessed dataset to:", task2_output_path)


Saved preprocessed dataset to: ../data/processed/preprocessed_texts.csv
